In [6]:
class Value:
    def __init__(self, data, prev=()):
        self.data = data
        self.grad = 0
        self.prev = prev
        self._backward = lambda:None
    
    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for n in v.prev:
                    build_topo(n)
                topo.append(v)
                
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)        
        out = Value(self.data + other.data, (self, other))
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1    
    
    def __sub__(self, other):
        return self + (-other)
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)        
        out = Value(self.data * other.data, (self, other))
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data ** other.data, (self,))       
        def _backward():
            self.grad += (other.data * self.data ** (other.data - 1)) * out.grad
        out._backward = _backward
        return out            

In [7]:
import random

class LinearLayer:
    def __init__(self, in_feat, out_feat):
        self.W = [[Value(random.uniform(-1, 1)) for _ in range(out_feat)] for _ in range(in_feat)]
        self.bias = [Value(0) for _ in range(out_feat)]
        self.shape = (in_feat, out_feat)
    
    def forward(self, X):
        out = [[Value(0) for _ in range(self.shape[1])] for _ in range(len(X))]
        
        for X_row_idx, row in enumerate(X):
            for W_col_idx in range(self.shape[1]):                
                for W_row_idx in range(self.shape[0]):            
                    out[X_row_idx][W_col_idx] += self.W[W_row_idx][W_col_idx] * row[W_row_idx] 
            
        for row in out:
            for i in range(self.shape[1]):
                row[i] += self.bias[i]              
                              
        return out
    
    def zero_grad(self):
        for row in self.W:
            for p in row:
                p.grad = 0
        for p in self.bias:
            p.grad = 0

In [8]:
lin = LinearLayer(2, 1)

In [9]:
out = lin.forward([[Value(4), Value(3)]])

In [15]:
mults = [1,2,3,4,5,6,7,8,9]
for _ in range(1000):
    lin.zero_grad()
    x = Value(random.choice(mults))
    y = Value(random.choice(mults))
    out = lin.forward([[x, y]])
    y_pred = 0
    for inner in out:
        for inner_inner in inner:
            y_pred = inner_inner
    y_true = x * y    
    loss = y_true**2 - y_pred**2
    print(loss.data) 
    loss.backward()
    
    lr = 1e-3
    for row in lin.W:
        for p in row:
            p += lr * (-p.grad)
    for p in lin.bias:
        p += lr * (-p.grad)        

959.7652866493098
713.6650547550424
117.01063870235835
2.4075866099844605
0.5650642765110696
355.978272604416
2789.0747599368133
222.83352598587305
525.0374897537146
1673.4761126532337
728.4863147540439
47.94132166232745
1222.4082796975135
-0.01466958441813837
-0.01466958441813837
728.4863147540439
216.77545582929943
379.80555753535344
1520.9588465160853
242.8111022557168
6.684052150697987
728.4863147540439
7.7083814964682595
277.39404225338694
1520.9588465160853
287.8679737402368
-1.3620568570546352
45.58556798809386
573.1773843529267
3022.7432551631764
1193.2148887903893
728.4863147540439
7.7083814964682595
959.7652866493098
117.01063870235835
3829.0507272394316
5030.390715694988
834.8000972547078
839.740517254375
1155.6676713087613
-1.0729046258829351
25.736822916972862
242.8111022557168
11.344162457603058
3843.4228581475545
839.740517254375
189.82771037656946
669.6682794898602
277.39404225338694
6.684052150697987
0.5650642765110696
287.8679737402368
3022.7432551631764
45.5855679880